# ValuArt – NFT Pricing Model
> **Predicting NFT sale prices from collection stats, rarity signals, and social engagement.**

This notebook walks through the full ML pipeline in a single, linear flow:
1. Data generation / loading
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Preprocessing
5. Model training & cross-validation
6. Hold-out evaluation & diagnostic plots

---

In [ ]:
import sys
import os

# Make sure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Libraries loaded ✓")

## 1  Load Data

In [ ]:
from src.data_loader import load_data

df = load_data()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0]

## 2  Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sale price distribution (raw)
axes[0].hist(df["sale_price"], bins=60, color="#2196F3", edgecolor="white")
axes[0].set_title("Sale Price Distribution (raw)", fontweight="bold")
axes[0].set_xlabel("Sale Price (ETH)")
axes[0].set_ylabel("Count")

# Sale price distribution (log)
axes[1].hist(np.log1p(df["sale_price"]), bins=60, color="#FF7043", edgecolor="white")
axes[1].set_title("Sale Price Distribution (log1p)", fontweight="bold")
axes[1].set_xlabel("log1p(Sale Price)")

plt.tight_layout()
plt.savefig("../outputs/plots/eda_price_dist.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved eda_price_dist.png")

In [ ]:
# Blockchain distribution
bc_counts = df["blockchain"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bc_counts.plot(kind="bar", ax=ax, color=["#1565C0","#6A1B9A","#00838F"], edgecolor="white")
ax.set_title("Listings by Blockchain", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../outputs/plots/eda_blockchain.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap (numeric columns)
import matplotlib.pyplot as plt

num_cols = [
    "rarity_score", "trait_count", "past_avg_price", "floor_price",
    "sales_last_7d", "sales_last_30d", "twitter_followers", "discord_members",
    "engagement_rate", "listing_count", "days_since_mint", "sale_price"
]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(num_cols, fontsize=9)
ax.set_title("Feature Correlation Matrix", fontweight="bold", fontsize=12)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig("../outputs/plots/eda_correlation.png", dpi=130, bbox_inches="tight")
plt.show()

## 3  Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features

df_eng = engineer_features(df)
print(f"Shape after engineering: {df_eng.shape}")

new_feats = [
    "price_to_floor_ratio", "supply_demand_ratio", "sales_velocity",
    "social_score", "engagement_weighted", "log_floor_price",
    "log_past_avg_price", "age_bucket"
]
df_eng[new_feats].describe().round(3)

## 4  Preprocessing & Train/Test Split

In [ ]:
from src.preprocess import NFTPreprocessor, split_data

train_df, test_df = split_data(df_eng, test_size=0.20, random_state=42)

preprocessor = NFTPreprocessor()
X_train, y_train = preprocessor.fit_transform(train_df)
X_test,  y_test  = preprocessor.transform(test_df)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
X_train.head(3)

## 5  Cross-Validation

In [ ]:
from src.train import cross_validate_models, train_final_models, save_best_model

cv_results = cross_validate_models(X_train, y_train)
cv_results

In [ ]:
# Bar chart of CV R² by model
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#1565C0", "#2E7D32", "#C62828"]
bars = ax.bar(cv_results["model"], cv_results["cv_r2"], color=colors, edgecolor="white", width=0.5)
ax.errorbar(
    cv_results["model"], cv_results["cv_r2"],
    yerr=cv_results["cv_r2_std"], fmt="none", color="black", capsize=6, linewidth=1.5
)
for bar, val in zip(bars, cv_results["cv_r2"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title("5-Fold Cross-Validated R² by Model", fontweight="bold")
ax.set_ylabel("CV R²")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/plots/cv_r2_comparison.png", dpi=130, bbox_inches="tight")
plt.show()

## 6  Train Final Models & Evaluate on Hold-Out Set

In [ ]:
fitted_models = train_final_models(X_train, y_train)
best_name, _  = save_best_model(fitted_models, cv_results)
print(f"Best model: {best_name}")

In [ ]:
from src.evaluate import run_evaluation

results = run_evaluation(fitted_models, cv_results, X_test, y_test, best_name)
results

## 7  Diagnostic Plots

In [ ]:
from IPython.display import Image, display
import os

for fname in ["feature_importance.png", "actual_vs_predicted.png", "residuals.png"]:
    path = os.path.join("..", "outputs", "plots", fname)
    if os.path.exists(path):
        print(f"── {fname} ──")
        display(Image(filename=path, width=750))

## 8  Single-Row Prediction Demo

In [ ]:
import joblib
from src.predict import load_model, predict_from_df

best_model = load_model()

# Construct a hypothetical new NFT listing
new_listing = pd.DataFrame([{
    "collection_name":   "Bored Ape YC",
    "creator":           "YugaLabs",
    "rarity_score":      120.5,
    "trait_count":       7,
    "past_avg_price":    85.0,
    "floor_price":       72.0,
    "sales_last_7d":     45,
    "sales_last_30d":    180,
    "twitter_followers": 850_000,
    "discord_members":   210_000,
    "engagement_rate":   0.042,
    "listing_count":     320,
    "days_since_mint":   730,
    "blockchain":        "ethereum",
}])

preds = predict_from_df(best_model, preprocessor, engineer_features(new_listing))
print(f"Predicted sale price: {preds.values[0]:.4f} ETH")

---
**End of Demo**

The full pipeline can also be run headlessly via:
```bash
python main.py
```